# <font color="#418FDE" size="6.5" uppercase>**KNN and Trees**</font>

>Last update: 20260709.
    
By the end of this Lecture, you will be able to:
- Implement k-nearest neighbors classification using manual distance calculations. 
- Explain how simple decision tree splits create interpretable engineering rules. 
- Compare logistic regression, k-nearest neighbors, and simple tree rules using test metrics. 


## **1. KNN From Scratch**

### **1.1. Distance Calculations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_01_01.jpg?v=1783618572" width="250">



>* Compare new cases with labeled examples
>* Nearest distances guide KNN classification

>* Feature scales can distort KNN distances
>* Normalize data to compare features fairly

>* Features place observations in distance space
>* Good features make neighbors meaningful



In [ ]:
#@title Python Code - Distance Calculations

# Manual distances reveal KNN similarity.
# Feature scaling changes nearest neighbors.
# Tiny engineering data keeps output readable.

import math
import numpy as np
import pandas as pd

# Store labeled machine observations.
data = pd.DataFrame({
    "temperature_c": [62, 65, 80, 83],
    "vibration_mm_s": [2.1, 2.4, 6.8, 7.2],
    "label": ["normal", "normal", "risk", "risk"]
})

# Define one new machine reading.
new_point = {"temperature_c": 78, "vibration_mm_s": 6.0}
features = ["temperature_c", "vibration_mm_s"]

# Validate the tiny table shape.
assert len(data) == 4 and len(features) == 2
assert all(name in data.columns for name in features)

# Compute Euclidean distance manually.
def euclidean_distance(row, point, columns):
    squared_diffs = []
    for column in columns:
        difference = row[column] - point[column]
        squared_diffs.append(difference ** 2)

    return math.sqrt(sum(squared_diffs))

# Add raw distances to observations.
data["raw_distance"] = data.apply(
    lambda row: euclidean_distance(row, new_point, features),
    axis=1
)

# Standardize features using training data.
means = data[features].mean()
stds = data[features].std(ddof=0)
scaled_data = data[features].subtract(means).divide(stds)

# Scale the new point consistently.
scaled_point = {
    column: (new_point[column] - means[column]) / stds[column]
    for column in features
}

# Compute distances after standardization.
data["scaled_distance"] = scaled_data.apply(
    lambda row: euclidean_distance(row, scaled_point, features),
    axis=1
)

# Find nearest neighbors for comparison.
raw_neighbor = data.sort_values("raw_distance").iloc[0]
scaled_neighbor = data.sort_values("scaled_distance").iloc[0]

# Print compact teaching results.
print("New machine: 78 C, 6.0 mm/s vibration")
print(f"Nearest using raw units: {raw_neighbor['label']}")
print(f"Raw distance: {raw_neighbor['raw_distance']:.2f}")
print(f"Nearest after scaling: {scaled_neighbor['label']}")
print(f"Scaled distance: {scaled_neighbor['scaled_distance']:.2f}")
print("Scaling helps each feature contribute more fairly.")

# Plot the small feature space.
colors = data["label"].map({"normal": "tab:blue", "risk": "tab:red"})
plt = __import__("matplotlib.pyplot")
plt.pyplot.figure(figsize=(6, 4))

# Draw known machines and new reading.
plt.pyplot.scatter(data["temperature_c"], data["vibration_mm_s"], c=colors, s=90)
plt.pyplot.scatter(new_point["temperature_c"], new_point["vibration_mm_s"], c="black", marker="X", s=120)
plt.pyplot.xlabel("Temperature, degrees Celsius")
plt.pyplot.ylabel("Vibration, millimeters per second")

# Add a concise plot title.
plt.pyplot.title("KNN distance idea for machine condition")
plt.pyplot.grid(True, alpha=0.3)
plt.pyplot.show()



### **1.2. Distance Calculation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_01_02.jpg?v=1783618576" width="250">



>* KNN compares new cases to known examples
>* Closest examples guide the predicted class

>* Compare features and rank nearest observations
>* Scale features to avoid misleading distances

>* Features place observations in comparison space
>* Good distances identify truly similar neighbors



In [ ]:
#@title Python Code - Distance Calculation

# Manual distances power nearest neighbor classification.
# Feature scaling changes engineering similarity comparisons.
# This example uses tiny machine readings.

import math
import pandas as pd

# Store labeled training readings in metric units.
training_rows = [
    {"temp_c": 62, "vibration_mm_s": 1.8, "label": "normal"},
    {"temp_c": 65, "vibration_mm_s": 2.1, "label": "normal"},

    {"temp_c": 88, "vibration_mm_s": 6.8, "label": "warning"},
    {"temp_c": 92, "vibration_mm_s": 7.4, "label": "warning"},
]

# Create a small table for convenient calculations.
train_df = pd.DataFrame(training_rows)
new_reading = {"temp_c": 70, "vibration_mm_s": 2.8}

# Validate the tiny dataset before distance calculations.
required_cols = {"temp_c", "vibration_mm_s", "label"}
assert required_cols.issubset(train_df.columns)
assert len(train_df) >= 3

# Define Euclidean distance using explicit feature differences.
def euclidean_distance(row, point):
    temp_diff = row["temp_c"] - point["temp_c"]
    vib_diff = row["vibration_mm_s"] - point["vibration_mm_s"]

    squared_sum = temp_diff**2 + vib_diff**2
    return math.sqrt(squared_sum)

# Calculate distance from the new reading to each example.
train_df["distance"] = train_df.apply(
    lambda row: euclidean_distance(row, new_reading), axis=1
)

# Rank examples from closest to farthest neighbor.
ranked_df = train_df.sort_values("distance").reset_index(drop=True)
k_value = 3

# Use the nearest labels for a simple majority vote.
nearest_labels = ranked_df.head(k_value)["label"].tolist()
prediction = max(set(nearest_labels), key=nearest_labels.count)

# Print compact results for classroom discussion.
print("New reading:", new_reading)
print("Nearest labels:", nearest_labels)
print("Predicted class:", prediction)
print("Closest distances:")

# Show only the nearest rows to avoid excessive output.
summary_cols = ["temp_c", "vibration_mm_s", "label", "distance"]
print(ranked_df[summary_cols].head(k_value).round(2).to_string(index=False))



### **1.3. Vote Among Neighbors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_01_03.jpg?v=1783618574" width="250">



>* Nearest examples vote on the new class
>* Most common neighbor label becomes prediction

>* Nearest labels decide the predicted class
>* k controls noise sensitivity and smoothing

>* Sort neighbors, count labels, resolve ties
>* Closer neighbors can receive stronger votes



In [ ]:
#@title Python Code - Vote Among Neighbors

# Demonstrate neighbor voting from scratch.
# Use tiny civil engineering examples.
# Keep output short and readable.

import math
from collections import Counter

# Store training cases as compact dictionaries.
training_cases = [
    {"name": "A", "features": (2.0, 8.0), "label": "low risk"},
    {"name": "B", "features": (3.0, 7.0), "label": "low risk"},

    {"name": "C", "features": (8.0, 3.0), "label": "high risk"},
    {"name": "D", "features": (7.0, 4.0), "label": "high risk"},

    {"name": "E", "features": (4.0, 6.0), "label": "low risk"},
]

# Define the new inspection case.
new_case = (5.0, 5.5)
k_neighbors = 3

# Validate the small example before voting.
assert len(training_cases) >= k_neighbors, "Need at least k cases."
assert len(new_case) == len(training_cases[0]["features"]), "Feature sizes differ."

# Compute Euclidean distance manually.
def euclidean_distance(point_a, point_b):
    squared_differences = []

    for value_a, value_b in zip(point_a, point_b):
        squared_differences.append((value_a - value_b) ** 2)

    return math.sqrt(sum(squared_differences))

# Attach distances to each training case.
distances = []
for case in training_cases:
    distance = euclidean_distance(new_case, case["features"])

    distances.append((distance, case["name"], case["label"]))

# Sort cases from closest to farthest.
sorted_distances = sorted(distances, key=lambda item: item[0])
neighbors = sorted_distances[:k_neighbors]

# Count labels among selected neighbors.
neighbor_labels = [item[2] for item in neighbors]
vote_counts = Counter(neighbor_labels)

# Choose the label with most votes.
predicted_label = vote_counts.most_common(1)[0][0]

# Print a compact teaching summary.
print("New case features: crack=5.0 mm, drainage=5.5 score")
print("Nearest neighbors:")

for distance, name, label in neighbors:
    print(f"Case {name}: distance={distance:.2f}, label={label}")

print(f"Vote counts: {dict(vote_counts)}")
print(f"Predicted class: {predicted_label}")



## **2. Simple Tree Rules**

### **2.1. Tree Split Rules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_02_01.jpg?v=1783618570" width="250">



>* Splits ask simple feature-based questions
>* Branches form readable diagnostic decision paths

>* Splits create cleaner same-class groups
>* Rule paths reveal feature interactions

>* Tree rules connect data to engineering decisions
>* Validate simple splits before trusting them



In [ ]:
#@title Python Code - Tree Split Rules

# Simple tree rules support engineering interpretation.
# Tiny data keeps the example transparent.
# Manual splits show readable classification logic.

import pandas as pd
import matplotlib.pyplot as plt

# Create a small inspection dataset.
data = pd.DataFrame({
    "roughness_um": [2.1, 3.4, 4.8, 5.2, 6.1, 7.0, 3.0, 5.8],
    "curing_hours": [18, 20, 22, 30, 28, 35, 32, 24],
    "failed": [0, 0, 0, 1, 1, 1, 0, 1],
})

# Define two simple tree split rules.
rough_rule = data["roughness_um"] > 5.0
cure_rule = data["curing_hours"] > 26

# Combine rules like a shallow decision tree.
data["predicted_fail"] = (rough_rule & cure_rule).astype(int)

# Count correct and incorrect classifications.
correct_count = int((data["failed"] == data["predicted_fail"]).sum())
total_count = int(len(data))
accuracy_value = correct_count / total_count

# Print compact teaching results.
print("Rule 1: roughness > 5.0 micrometers.")
print("Rule 2: curing time > 26 hours.")
print("Predict fail only when both rules are true.")
print(f"Accuracy on tiny example: {accuracy_value:.2f}")

# Prepare colors for actual inspection outcomes.
color_map = data["failed"].map({0: "steelblue", 1: "crimson"})

# Plot points and split thresholds.
plt.figure(figsize=(6, 4))
plt.scatter(data["roughness_um"], data["curing_hours"], c=color_map, s=80)
plt.axvline(5.0, color="black", linestyle="--", label="roughness split")
plt.axhline(26, color="gray", linestyle="--", label="curing split")

# Label the engineering axes clearly.
plt.xlabel("Surface roughness, micrometers")
plt.ylabel("Curing time, hours")
plt.title("Simple tree split rules for part failure")
plt.legend(loc="lower right")

# Display the single teaching plot.
plt.show()



### **2.2. Interpretable Split Rules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_02_02.jpg?v=1783618566" width="250">



>* Tree splits use clear threshold checks
>* Decision paths become inspectable engineering rules

>* Familiar features make splits easier to explain
>* Thresholds show data-driven decision boundaries

>* Clear rules support trust and action
>* Simple splits need validation and refinement



In [ ]:
#@title Python Code - Interpretable Split Rules

# Simple tree rules support engineering decisions.
# Each split becomes readable plain language.
# Tiny data keeps the example transparent.

import numpy as np
import matplotlib.pyplot as plt

# Store bridge inspection measurements.
crack_mm = np.array([0.2, 0.4, 0.7, 1.1, 1.4, 1.8, 2.2, 2.6])
traffic_kips = np.array([12, 18, 25, 22, 35, 30, 45, 50])
priority = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Define one interpretable split rule.
crack_limit = 1.25
traffic_limit = 28

# Apply the rule step by step.
wide_crack = crack_mm > crack_limit
heavy_traffic = traffic_kips > traffic_limit
predicted = (wide_crack & heavy_traffic).astype(int)

# Check that arrays align safely.
assert crack_mm.size == traffic_kips.size == priority.size
assert predicted.size == priority.size

# Compute a simple accuracy metric.
accuracy = np.mean(predicted == priority)
rule_text = "IF crack > 1.25 mm AND traffic > 28 kips THEN high priority"

# Print concise teaching output.
print("Interpretable split rule:")
print(rule_text)
print(f"Accuracy on tiny examples: {accuracy:.2f}")
print("Each branch uses measurable engineering quantities.")

# Plot the two split thresholds.
colors = np.where(priority == 1, "tomato", "steelblue")
plt.figure(figsize=(6, 4))
plt.scatter(crack_mm, traffic_kips, c=colors, s=80)

# Add readable threshold lines.
plt.axvline(crack_limit, color="black", linestyle="--")
plt.axhline(traffic_limit, color="black", linestyle="--")
plt.xlabel("Crack width, millimeters")
plt.ylabel("Traffic load, kips")

# Label the engineering decision region.
plt.title("Simple Tree Rule: Two Interpretable Splits")
plt.text(1.35, 42, "High-priority region", fontsize=10)
plt.tight_layout()
plt.show()



### **2.3. Readable Engineering Rules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_02_03.jpg?v=1783618568" width="250">



>* Tree splits turn measurements into practical conditions
>* Rules explain predictions in familiar engineering language

>* Use meaningful variables and clear thresholds
>* Link data patterns to practical decisions

>* Visible rules build trust and validation
>* Transparency supports fair, defensible decisions



In [ ]:
#@title Python Code - Readable Engineering Rules

# Simple tree rules support engineering decisions.
# Thresholds convert measurements into readable conditions.
# This example uses tiny inspection data.

import numpy as np
import pandas as pd

# Create small civil inspection records.
data = pd.DataFrame({
    "crack_mm": [1, 2, 4, 6, 8, 10, 12, 14],
    "deflection_in": [0.05, 0.08, 0.10, 0.18, 0.22, 0.30, 0.35, 0.40],
    "needs_repair": [0, 0, 0, 1, 1, 1, 1, 1]
})

# Define one readable tree rule.
def simple_tree_rule(crack_mm, deflection_in):
    if crack_mm >= 6:
        return 1
    if deflection_in >= 0.20:
        return 1
    return 0

# Apply the rule to every record.
predictions = []
for row in data.itertuples(index=False):
    predictions.append(simple_tree_rule(row.crack_mm, row.deflection_in))

data["predicted"] = predictions

# Compute compact classification metrics.
correct = int((data["needs_repair"] == data["predicted"]).sum())
total = int(len(data))
accuracy = correct / total

# Count confusion matrix entries.
tp = int(((data.needs_repair == 1) & (data.predicted == 1)).sum())
tn = int(((data.needs_repair == 0) & (data.predicted == 0)).sum())
fp = int(((data.needs_repair == 0) & (data.predicted == 1)).sum())
fn = int(((data.needs_repair == 1) & (data.predicted == 0)).sum())

# Print readable engineering interpretation.
print("Rule: repair if crack >= 6 mm or deflection >= 0.20 in")
print(f"Accuracy: {accuracy:.2f} using {total} inspection records")
print(f"Confusion counts: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print("Example path: crack 8 mm meets threshold, so repair is predicted")
print("Engineering value: the model explains its decision as thresholds")



## **3. Classifier Comparison**

### **3.1. Model Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_03_01.jpg?v=1783618578" width="250">



>* Use metrics to compare unseen test performance
>* Accuracy alone can hide costly errors

>* Precision checks correctness of positive predictions
>* Recall finds positives; F1 balances both

>* Models show different strengths on test data
>* Choose metrics matching the decision context



In [ ]:
#@title Python Code - Model Metrics

# Compare classifiers using shared test metrics.
# Civil examples use compact synthetic measurements.
# Built-in Python keeps calculations transparent.

import math
import random

random.seed(7)

# Create small inspection data with two features.
data = [
    (2.0, 18.0, 0), (2.5, 20.0, 0),
    (3.0, 22.0, 0), (3.5, 24.0, 0),

    (4.0, 28.0, 1), (4.5, 30.0, 1),
    (5.0, 34.0, 1), (5.5, 36.0, 1),

    (6.0, 40.0, 1), (1.8, 19.0, 0),
    (4.8, 27.0, 1), (3.2, 29.0, 0),
]

# Split examples into training and testing sets.
train = data[:8]
test = data[8:]

# Standardize features using training statistics only.
means = [sum(row[i] for row in train) / len(train) for i in range(2)]
spreads = [max(row[i] for row in train) - min(row[i] for row in train) for i in range(2)]

# Protect against accidental zero feature spread.
spreads = [value if value else 1.0 for value in spreads]

# Convert one row into scaled feature values.
def scaled(row):
    return [(row[i] - means[i]) / spreads[i] for i in range(2)]

# Logistic regression score uses fixed interpretable weights.
def logistic_predict(row):
    z = -0.4 + 2.2 * scaled(row)[0] + 1.4 * scaled(row)[1]
    return 1 if 1 / (1 + math.exp(-z)) >= 0.5 else 0

# KNN predicts from nearest scaled training examples.
def knn_predict(row, k=3):
    distances = []
    for item in train:
        d = math.dist(scaled(row), scaled(item))
        distances.append((d, item[2]))

    votes = [label for _, label in sorted(distances)[:k]]
    return 1 if sum(votes) >= 2 else 0

# Simple tree rule mimics an interpretable engineering threshold.
def tree_predict(row):
    crack_mm, load_kips, label = row
    return 1 if crack_mm >= 4.0 and load_kips >= 27.0 else 0

# Compute accuracy, precision, recall, and F1 score.
def metrics(predictor):
    pairs = [(predictor(row), row[2]) for row in test]
    tp = sum(1 for pred, true in pairs if pred == true == 1)
    tn = sum(1 for pred, true in pairs if pred == true == 0)

    fp = sum(1 for pred, true in pairs if pred == 1 and true == 0)
    fn = sum(1 for pred, true in pairs if pred == 0 and true == 1)
    accuracy = (tp + tn) / len(pairs)
    precision = tp / (tp + fp) if tp + fp else 0.0

    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return accuracy, precision, recall, f1

# Compare all models on the same held-out examples.
models = {
    "Logistic": logistic_predict,
    "KNN": knn_predict,
    "Tree rule": tree_predict,
}

print("Model       Accuracy  Precision  Recall  F1")
for name, predictor in models.items():
    values = metrics(predictor)
    print(f"{name:<10} {values[0]:>7.2f} {values[1]:>10.2f} {values[2]:>7.2f} {values[3]:>5.2f}")

print("Tree rule: defective if crack >= 4.0 mm and load >= 27 kips.")



### **3.2. Decision Tree Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_03_02.jpg?v=1783618580" width="250">



>* Tree rules need test-set validation
>* Use full metrics to reveal mistakes

>* Tree depth affects learning and overfitting
>* Test metrics show performance on new cases

>* Compare models by their different error patterns
>* Balance test performance with interpretability needs



In [ ]:
#@title Python Code - Decision Tree Metrics

# Compare classifiers using small engineering data.
# Decision tree metrics reveal practical mistakes.
# Built-in code avoids machine learning libraries.

import math
import random
import numpy as np

# Set deterministic randomness for repeatable results.
random.seed(7)
np.random.seed(7)

# Create tiny bridge inspection style data.
n = 80
age = np.random.uniform(5, 80, n)
crack = np.random.uniform(0.0, 1.2, n)

# Define risk using interpretable engineering thresholds.
noise = np.random.normal(0, 0.35, n)
score = 0.055 * age + 3.2 * crack + noise
risk = (score > 4.6).astype(int)

# Split data into train and test sets.
X = np.column_stack((age, crack))
y = risk
idx = np.random.permutation(n)
train_idx, test_idx = idx[:56], idx[56:]

# Standardize features using training statistics only.
mu = X[train_idx].mean(axis=0)
sigma = X[train_idx].std(axis=0) + 1e-9
Xz = (X - mu) / sigma

# Train logistic regression with simple gradient descent.
w = np.zeros(2)
b = 0.0
lr = 0.25

for step in range(500):
    z = Xz[train_idx] @ w + b
    p = 1 / (1 + np.exp(-z))
    err = p - y[train_idx]
    w -= lr * (Xz[train_idx].T @ err) / len(train_idx)
    b -= lr * err.mean()

# Predict with logistic regression probabilities.
logit_prob = 1 / (1 + np.exp(-(Xz[test_idx] @ w + b)))
logit_pred = (logit_prob >= 0.5).astype(int)

# Predict with manual three-nearest-neighbor distances.
def knn_predict(row, train_rows, train_labels, k=3):
    distances = np.sqrt(((train_rows - row) ** 2).sum(axis=1))
    nearest = np.argsort(distances)[:k]
    return int(train_labels[nearest].mean() >= 0.5)

knn_pred = np.array([
    knn_predict(row, Xz[train_idx], y[train_idx])
    for row in Xz[test_idx]
])

# Predict with a simple interpretable tree rule.
tree_pred = ((X[test_idx, 0] > 45) & (X[test_idx, 1] > 0.45)).astype(int)

# Compute compact classification metrics.
def metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    acc = (tp + tn) / len(y_true)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    return acc, prec, rec, spec, tp, fp, fn, tn

# Print concise test metrics for comparison.
print("Model comparison on held-out bridge cases")
print("Model        Acc  Prec Recall Spec  TP FP FN TN")

for name, pred in [("Logistic", logit_pred), ("KNN", knn_pred), ("Tree rule", tree_pred)]:
    vals = metrics(y[test_idx], pred)
    print(f"{name:10s} {vals[0]:.2f} {vals[1]:.2f} {vals[2]:.2f} {vals[3]:.2f}  {vals[4]} {vals[5]} {vals[6]} {vals[7]}")

# State the visible decision tree rule.
print("Tree rule: high risk if age > 45 years and crack > 0.45 inches")



### **3.3. Interpreting Test Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_03_03.jpg?v=1783618582" width="250">



>* Test data shows real-world generalization
>* Different classifiers fit different pattern types

>* Use precision and recall beyond accuracy
>* Choose models by acceptable error tradeoffs

>* Balance metrics with interpretability and practicality
>* Choose models that fit real consequences



In [ ]:
#@title Python Code - Interpreting Test Metrics

# Compare classifiers using held-out test metrics.
# Civil engineering examples need interpretable errors.
# Built-in Python keeps calculations transparent.

import math
import random

random.seed(7)

# Create tiny component inspection records.
data = [
    (0.9, 42, 0), (1.1, 45, 0),
    (1.4, 48, 0), (1.8, 52, 0),
    (2.2, 58, 1), (2.6, 61, 1),
    (3.0, 66, 1), (3.4, 70, 1),
    (1.7, 63, 1), (2.8, 50, 0),
    (3.2, 55, 1), (1.2, 68, 0),
]

# Split records into training and testing.
train = data[:8]
test = data[8:]

# Validate the small split sizes.
assert len(train) >= 4 and len(test) >= 2

# Standardize features using training statistics.
means = [sum(row[i] for row in train) / len(train) for i in range(2)]
spreads = [max(row[i] for row in train) - min(row[i] for row in train) for i in range(2)]

# Avoid division by zero safely.
spreads = [value if value else 1.0 for value in spreads]

def scaled_features(row):
    return [(row[i] - means[i]) / spreads[i] for i in range(2)]

# Train logistic regression manually.
weights = [0.0, 0.0]
bias = 0.0
rate = 0.8

for epoch in range(120):
    for row in train:
        x = scaled_features(row)
        z = bias + sum(weights[i] * x[i] for i in range(2))
        pred = 1 / (1 + math.exp(-z))
        error = pred - row[2]
        weights = [weights[i] - rate * error * x[i] for i in range(2)]
        bias = bias - rate * error

# Predict with logistic regression.
def logistic_predict(row):
    x = scaled_features(row)
    z = bias + sum(weights[i] * x[i] for i in range(2))
    return int((1 / (1 + math.exp(-z))) >= 0.5)

# Predict with three nearest neighbors.
def knn_predict(row, k=3):
    x = scaled_features(row)
    distances = []
    for item in train:
        y = scaled_features(item)
        dist = math.sqrt(sum((x[i] - y[i]) ** 2 for i in range(2)))
        distances.append((dist, item[2]))
    votes = [label for _, label in sorted(distances)[:k]]
    return int(sum(votes) >= (k / 2))

# Predict with a simple engineering rule.
def tree_rule_predict(row):
    vibration, temperature = row[0], row[1]
    return int(vibration >= 2.0 and temperature >= 55)

# Compute common test metrics.
def metrics(predictor):
    pairs = [(predictor(row), row[2]) for row in test]
    tp = sum(pred == 1 and true == 1 for pred, true in pairs)
    tn = sum(pred == 0 and true == 0 for pred, true in pairs)
    fp = sum(pred == 1 and true == 0 for pred, true in pairs)
    fn = sum(pred == 0 and true == 1 for pred, true in pairs)
    accuracy = (tp + tn) / len(pairs)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return accuracy, precision, recall

# Compare models on identical test records.
models = {
    "Logistic regression": logistic_predict,
    "K-nearest neighbors": knn_predict,
    "Simple tree rule": tree_rule_predict,
}

print("Test set: predict component failure from vibration and temperature")
print("Model                 Accuracy  Precision  Recall")

for name, predictor in models.items():
    acc, prec, rec = metrics(predictor)
    print(f"{name:<22} {acc:>6.2f}     {prec:>6.2f}   {rec:>6.2f}")

print("Interpretation: choose metrics matching the cost of each error.")



# <font color="#418FDE" size="6.5" uppercase>**KNN and Trees**</font>


In this lecture, you learned to:
- Implement k-nearest neighbors classification using manual distance calculations. 
- Explain how simple decision tree splits create interpretable engineering rules. 
- Compare logistic regression, k-nearest neighbors, and simple tree rules using test metrics. 

In the next Module (Module 6), we will go over 'None'